# 실습 - 제로샷 분류

**야생동물 카메라 트랩 이미지의 종 분류**

제로샷(zero-shot) 분류는 해당 범주의 **학습 예시를 하나도 사용하지 않고**, 분류 범주를 추론 시점에 텍스트로 지정하여 분류하는 방식이다.

본 실습에서는 공개 데이터셋의 카메라 트랩 이미지에 대해, 모델을 전혀 학습시키지 않은 상태로 동물의 종을 예측하고 정답과 비교한다.

**실습 순서**

- 데이터셋 불러오기
- 데이터 구조 확인
- 이미지와 정답 확인
- 제로샷 분류 수행
- 예측과 정답 비교
- 후보 문구 변경 실험

---
## 0. 환경 준비

실습에 필요한 라이브러리를 불러온다.

- `datasets` — Hub 데이터셋 내려받기
- `transformers` — 사전학습 모델로 추론 수행

In [ ]:
from datasets import load_dataset
from transformers import pipeline

print("준비 완료")

---
## 1. 데이터셋 불러오기

`load_dataset()` 으로 Hub의 카메라 트랩 데이터셋을 내려받는다.

- 데이터셋 이름 — `imageomics/IDLE-OO-Camera-Traps`
- 구성 — 카메라 트랩 이미지 2,586장, 종별로 균형 있게 구성
- 출처 — LILA BC (Labeled Information Library of Alexandria: Biology and Conservation)

**split 지정 방법**

이 데이터셋의 구간은 `test` 하나이다.

- `"test"` — 전체 2,586장
- `"test[:100]"` — 앞 100장만 내려받기
- `"test[:5%]"` — 비율로 지정

실습에서는 시간 단축을 위해 앞 100장만 사용한다.

In [ ]:
ds = load_dataset(
    "imageomics/IDLE-OO-Camera-Traps",
    split="test[:100]"
)

---
## 2. 데이터 구조 확인

불러온 데이터가 어떤 항목으로 구성되는지 확인한다.

- `features` — 각 항목의 구성 열
- `num_rows` — 불러온 이미지 수

열 이름은 데이터셋마다 다르므로 **구조 확인은 필수 절차**이다.

In [ ]:
ds

주요 열은 다음과 같다.

- `image` — 카메라 트랩 이미지
- `common_name` — 일반명 (예: cheetah)
- `scientific_name` — 학명 (예: acinonyx jubatus)
- 그 외 — 분류 계층 (kingdom, phylum, cls, order, family, genus, species)

본 실습에서는 `image` 와 `common_name` 을 사용한다.

---
## 3. 이미지와 정답 확인

### 3-1. 이미지 확인

`ds[1]` 으로 첫 번째 항목을 선택하고, 이미지 열만 꺼내 화면에 표시한다.

In [ ]:
ds[1]["image"]

### 3-2. 정답 확인

각 이미지에는 정답 종 이름이 함께 저장되어 있다.

In [ ]:
print("정답:", ds[0]["common_name"])
print("학명:", ds[0]["scientific_name"])

### 3-3. 후보 목록 생성

제로샷 분류에는 **후보 목록**이 필요하다. 데이터셋에 등장하는 종 이름에서 중복을 제거하여 후보 목록을 만든다.

- `set` — 중복 제거
- `sorted` — 이름순 정렬

이 목록을 이후 분류에서 후보로 사용한다.

In [ ]:
labels = sorted(set(ds["common_name"]))

print("후보 수:", len(labels))
print(labels)

---
## 4. 제로샷 분류 수행

### 4-1. 모델 준비

작업 이름을 `zero-shot-image-classification` 으로 지정한다. 앞선 실습과 **동일한 구조**이며, 작업 이름과 모델만 바뀐다.

- 모델 — `google/siglip-base-patch16-224`
- 이미지와 텍스트를 함께 학습한 모델이므로, 텍스트 후보와 이미지를 비교할 수 있다

In [ ]:
clf = pipeline(
    task="zero-shot-image-classification",
    model="google/siglip-base-patch16-224"
)

### 4-2. 분류 실행

이미지와 후보 목록을 함께 입력한다.

- `candidate_labels` — 후보 목록 지정
- 출력은 후보마다 라벨과 점수를 부여한 목록
- 점수가 높은 순으로 정렬

In [ ]:
result = clf(
    ds[0]["image"],
    candidate_labels=labels
)

result[:5]

---
## 5. 예측과 정답 비교

### 5-1. 개별 비교

점수가 가장 높은 예측을 정답과 나란히 확인한다.

- `result[0]` — 점수가 가장 높은 예측

In [ ]:
print("예측:", result[0]["label"])
print("정답:", ds[0]["common_name"])
print("점수:", round(result[0]["score"], 4))

### 5-2. 여러 장 비교

앞 20장에 대해 예측과 정답이 일치한 횟수를 집계한다.

- `select` — 지정한 개수만 선택
- 모델을 **전혀 학습시키지 않은** 상태의 성능

정확도는 1(100%)에 가까울수록 정확하다. 후보 수가 많을수록 난이도가 높아지므로, 무작위로 맞힐 확률과 비교하여 해석한다.

In [ ]:
n = 20
correct = 0

for item in ds.select(range(n)):
    pred = clf(item["image"], candidate_labels=labels)
    if pred[0]["label"] == item["common_name"]:
        correct += 1

print("정확도:", correct / n)
print("무작위 확률:", round(1 / len(labels), 4))

---
## 6. 후보 문구 변경 실험

후보를 **행동 문구**로 바꾸면, 같은 이미지와 같은 모델로 완전히 다른 판단을 수행한다.

이미지와 모델은 그대로 두고 **후보만 교체**한다는 점이 핵심이다.

In [ ]:
behaviors = [
    "a picture of an animal eating",
    "a picture of an animal moving",
    "a picture of an animal resting",
    "a camera trap picture of an animal eating",
    "a camera trap picture of an animal moving",
    "a camera trap picture of an animal resting",
]

ds[1]["image"]

In [ ]:
clf(ds[1]["image"], candidate_labels=behaviors)

### 6-1. 문구 형태 비교

후보를 단어만으로 제시할 때와 문장으로 제시할 때 결과가 어떻게 달라지는지 확인한다.

후보 문구의 구성 방식이 성능에 큰 영향을 준다는 점은 실제 연구에서도 확인된 사항이다.

In [ ]:
short = ["eating", "moving", "resting"]

print("단어:", clf(ds[0]["image"], candidate_labels=short)[0])
print("문장:", clf(ds[0]["image"], candidate_labels=behaviors)[0])

**주의**

행동에는 정답 라벨이 없으므로 성능을 수치로 검증할 수 없다. 사진을 직접 보며 판단해야 한다.

행동이 주석된 공개 이미지 데이터는 매우 드물다. 정답 라벨이 없으면 성능 검증도, 추가 학습도 어렵다는 점이 다음 주제인 **미세조정**으로 이어진다.

---
## 7. 정리

**제로샷 분류의 특징**

- 해당 범주의 학습 예시를 사용하지 않음
- 분류 범주를 추론 시점에 텍스트로 지정
- 모델 파라미터를 변경하지 않음

**실습에서 확인한 사항**

- 학습 없이도 상당한 수준의 종 분류가 가능
- 후보 목록에 없는 범주는 예측 불가
- 후보 문구의 표현 방식이 결과에 영향

**한계**

- 정답 라벨이 없으면 성능 검증 불가
- 후보 수가 많을수록 난이도 상승
- 데이터 특성이 학습 데이터와 다를수록 성능 저하

---

데이터 출처: imageomics/IDLE-OO-Camera-Traps (LILA BC 기반, Hugging Face Hub 공개)